# Setup: Abhaengigkeiten installieren

Diese Zelle bitte einmal ausfuehren, bevor das Training gestartet wird.

In [ ]:
# Installiert alle benoetigten Pakete fuer dieses Notebook
%pip install -U pip
%pip install -r extra_requirements.txt
%pip install speechbrain torch torchaudio tensorboard

In [ ]:
from pathlib import Path
import itertools
import torchaudio

# Optional fest setzen, falls Auto-Erkennung nicht passt:
# dataset_root = Path('/home/lugra002/speechbrain/data/noisy-vctk-16k')
dataset_root = None

required_subdirs = [
    'clean_trainset_28spk_wav_16k',
    'noisy_trainset_28spk_wav_16k',
    'clean_testset_wav_16k',
    'noisy_testset_wav_16k',
    'trainset_28spk_txt',
    'testset_txt',
]

def is_valid_voicebank_root(path: Path) -> bool:
    return path.is_dir() and all((path / d).exists() for d in required_subdirs)

project_dir = Path.cwd()
candidates = [
    project_dir / 'data',
    project_dir.parent / 'data',
    project_dir.parent.parent / 'data',
    Path.home() / 'speechbrain' / 'data',
    Path.home() / 'speechbrain' / 'data' / 'noisy-vctk-16k',
]

if dataset_root is None:
    for c in candidates:
        if is_valid_voicebank_root(c):
            dataset_root = c.resolve()
            break
        nested = c / 'noisy-vctk-16k'
        if is_valid_voicebank_root(nested):
            dataset_root = nested.resolve()
            break

if dataset_root is None:
    raise FileNotFoundError(
        'Kein gueltiger Voicebank-Datenordner gefunden. '
        'Bitte dataset_root manuell setzen.'
    )

print(f'Pruefe Datensatz unter: {dataset_root}')

# 1) Strukturcheck
missing = [d for d in required_subdirs if not (dataset_root / d).exists()]
if missing:
    raise FileNotFoundError(f'Fehlende Unterordner: {missing}')

train_clean = dataset_root / 'clean_trainset_28spk_wav_16k'
train_noisy = dataset_root / 'noisy_trainset_28spk_wav_16k'
test_clean = dataset_root / 'clean_testset_wav_16k'
test_noisy = dataset_root / 'noisy_testset_wav_16k'
train_txt = dataset_root / 'trainset_28spk_txt'
test_txt = dataset_root / 'testset_txt'

def stems_from_wavs(folder: Path):
    return {p.stem for p in folder.rglob('*.wav')}

def stems_from_txt(folder: Path):
    return {p.stem for p in folder.rglob('*.txt')}

train_clean_stems = stems_from_wavs(train_clean)
train_noisy_stems = stems_from_wavs(train_noisy)
test_clean_stems = stems_from_wavs(test_clean)
test_noisy_stems = stems_from_wavs(test_noisy)
train_txt_stems = stems_from_txt(train_txt)
test_txt_stems = stems_from_txt(test_txt)

# 2) Paired-Setup wie im Paper/Recipe: clean-noisy Paare fuer train/test
errors = []
if not train_clean_stems or not train_noisy_stems:
    errors.append('Train-WAVs fehlen oder sind leer.')
if not test_clean_stems or not test_noisy_stems:
    errors.append('Test-WAVs fehlen oder sind leer.')
if train_clean_stems != train_noisy_stems:
    only_clean = len(train_clean_stems - train_noisy_stems)
    only_noisy = len(train_noisy_stems - train_clean_stems)
    errors.append(
        f'Train clean/noisy nicht 1:1 gepaart (nur clean: {only_clean}, nur noisy: {only_noisy}).'
    )
if test_clean_stems != test_noisy_stems:
    only_clean = len(test_clean_stems - test_noisy_stems)
    only_noisy = len(test_noisy_stems - test_clean_stems)
    errors.append(
        f'Test clean/noisy nicht 1:1 gepaart (nur clean: {only_clean}, nur noisy: {only_noisy}).'
    )

# 3) Transkript-Abdeckung fuer die JSON-Erstellung (words/phones Felder)
if not train_noisy_stems.issubset(train_txt_stems):
    miss = len(train_noisy_stems - train_txt_stems)
    errors.append(f'Es fehlen {miss} Train-Transkripte (.txt).')
if not test_noisy_stems.issubset(test_txt_stems):
    miss = len(test_noisy_stems - test_txt_stems)
    errors.append(f'Es fehlen {miss} Test-Transkripte (.txt).')

# 4) Audioformat-Check: 16 kHz, mono, nicht leer
def wav_paths(folder: Path):
    return sorted(folder.rglob('*.wav'))

all_wavs = (
    wav_paths(train_clean) + wav_paths(train_noisy) +
    wav_paths(test_clean) + wav_paths(test_noisy)
)

check_all_audio = False  # True fuer Vollpruefung aller WAVs
max_audio_checks = len(all_wavs) if check_all_audio else min(500, len(all_wavs))

bad_sr = 0
bad_channels = 0
bad_frames = 0
for wav_file in itertools.islice(all_wavs, max_audio_checks):
    info = torchaudio.info(str(wav_file))
    if info.sample_rate != 16000:
        bad_sr += 1
    if info.num_channels != 1:
        bad_channels += 1
    if info.num_frames <= 0:
        bad_frames += 1

if bad_sr:
    errors.append(f'{bad_sr} gepruefte WAVs haben nicht 16kHz.')
if bad_channels:
    errors.append(f'{bad_channels} gepruefte WAVs sind nicht mono.')
if bad_frames:
    errors.append(f'{bad_frames} gepruefte WAVs sind leer/defekt (0 Frames).')

# 5) Sprecherstatistik (VoiceBank+DEMAND typischerweise 28 Train-Sprecher, 2 Test-Sprecher)
def speaker_ids(stems):
    return {s.split('_')[0] for s in stems if '_' in s}

n_train_speakers = len(speaker_ids(train_noisy_stems))
n_test_speakers = len(speaker_ids(test_noisy_stems))

print('--- Zusammenfassung ---')
print(f'Train-Paare: {len(train_noisy_stems)}')
print(f'Test-Paare:  {len(test_noisy_stems)}')
print(f'Train-Sprecher: {n_train_speakers} (erwartet typischerweise 28)')
print(f'Test-Sprecher:  {n_test_speakers} (erwartet typischerweise 2)')
print(f'Audio-Dateien geprueft: {max_audio_checks}/{len(all_wavs)}')

if errors:
    print('\nFEHLER:')
    for e in errors:
        print('-', e)
    raise RuntimeError('Datensatzformat ungueltig. Details siehe oben.')

print('OK: Datensatzformat ist fuer die MetricGAN+-Pipeline gueltig.')

# Einzelrun mit Ablations-Config

Diese Zelle startet `train.py` mit einer Datei aus `hparams/ablations_core` (nicht mit `hparams/train.yaml`).

In [ ]:
from pathlib import Path
import subprocess
import sys

project_dir = Path.cwd()
script_path = project_dir / 'train.py'
ablations_dir = project_dir / 'hparams' / 'ablations_core'

# Einzelrun: waehle eine Ablationsdatei (statt hparams/train.yaml)
hparams_name = 'train_fixed_l1_n10_e120.yaml'
hparams_file = ablations_dir / hparams_name

required_subdirs = [
    'clean_trainset_28spk_wav_16k',
    'noisy_trainset_28spk_wav_16k',
    'clean_testset_wav_16k',
    'noisy_testset_wav_16k',
    'trainset_28spk_txt',
    'testset_txt',
]

def is_valid_voicebank_root(path: Path) -> bool:
    return path.is_dir() and all((path / d).exists() for d in required_subdirs)

# Optional manuell setzen, z.B.:
# data_folder = '/home/lugra002/speechbrain/data/noisy-vctk-16k'
data_folder = None

candidates = [
    project_dir / 'data',
    project_dir.parent / 'data',
    project_dir.parent.parent / 'data',
    Path.home() / 'speechbrain' / 'data',
    Path.home() / 'speechbrain' / 'data' / 'noisy-vctk-16k',
]

if data_folder is None:
    for c in candidates:
        if is_valid_voicebank_root(c):
            data_folder = c
            break
        nested = c / 'noisy-vctk-16k'
        if is_valid_voicebank_root(nested):
            data_folder = nested
            break
else:
    data_folder = Path(data_folder).expanduser()

if data_folder is None:
    raise FileNotFoundError(
        'Kein gueltiger Voicebank-Datenordner gefunden. '
        'Setze data_folder explizit auf einen Pfad mit den Unterordnern: '
        + ', '.join(required_subdirs)
    )

if not is_valid_voicebank_root(data_folder):
    raise FileNotFoundError(
        f'Ungueltiger data_folder: {data_folder}. Erwartete Unterordner: '
        + ', '.join(required_subdirs)
    )

data_folder = str(data_folder.resolve())

if not script_path.exists():
    raise FileNotFoundError(f'Nicht gefunden: {script_path}')
if not hparams_file.exists():
    raise FileNotFoundError(f'Nicht gefunden: {hparams_file}')

cmd = [
    sys.executable,
    str(script_path),
    str(hparams_file),
    f'data_folder={data_folder}',
]

print('Verwende data_folder:', data_folder)
print('Verwende Ablation:', hparams_file.name)
print('Starte:', ' '.join(cmd))
subprocess.run(cmd, check=True, cwd=project_dir)

# Core-Ablations iterativ starten

Diese Zellen laufen automatisch ueber alle YAMLs in `hparams/ablations_core`.

- Optional kannst du mehrere Seeds angeben.
- `dry_run=True` zeigt nur die Befehle, ohne Training zu starten.
- `max_runs` begrenzt die Anzahl gestarteter Jobs zum Testen.

In [ ]:
from pathlib import Path
import subprocess
import sys

project_dir = Path.cwd()
script_path = project_dir / 'train.py'
hparams_root = project_dir / 'hparams'

# Falls du nur einzelne Konfigurationen laufen lassen willst, hier Dateinamen eintragen.
# Beispiel: selected_files = {'train_fixed_l1_n10_e120.yaml', 'train.yaml'}
selected_files = None

# Mehrere Seeds fuer robustere Aussagen
seeds = [4234, 5234, 6234]

# Optional: Epochenzahl zentral ueberschreiben (None = YAML-Wert behalten)
override_epochs = 220

# True: nur Kommandos anzeigen; False: wirklich starten
dry_run = True

# Optional zum Antesten limitieren, z.B. 2; None = alle
max_runs = None

required_subdirs = [
    'clean_trainset_28spk_wav_16k',
    'noisy_trainset_28spk_wav_16k',
    'clean_testset_wav_16k',
    'noisy_testset_wav_16k',
    'trainset_28spk_txt',
    'testset_txt',
]

def is_valid_voicebank_root(path: Path) -> bool:
    return path.is_dir() and all((path / d).exists() for d in required_subdirs)

# Optional manuell setzen, z.B.:
# data_folder = '/home/lugra002/speechbrain/data/noisy-vctk-16k'
data_folder = None

candidates = [
    project_dir / 'data',
    project_dir.parent / 'data',
    project_dir.parent.parent / 'data',
    Path.home() / 'speechbrain' / 'data',
    Path.home() / 'speechbrain' / 'data' / 'noisy-vctk-16k',
]

if data_folder is None:
    for c in candidates:
        if is_valid_voicebank_root(c):
            data_folder = c
            break
        nested = c / 'noisy-vctk-16k'
        if is_valid_voicebank_root(nested):
            data_folder = nested
            break
else:
    data_folder = Path(data_folder).expanduser()

if data_folder is None:
    raise FileNotFoundError(
        'Kein gueltiger Voicebank-Datenordner gefunden. '
        'Setze data_folder explizit auf einen Pfad mit den Unterordnern: '
        + ', '.join(required_subdirs)
    )

if not is_valid_voicebank_root(data_folder):
    raise FileNotFoundError(
        f'Ungueltiger data_folder: {data_folder}. Erwartete Unterordner: '
        + ', '.join(required_subdirs)
    )

data_folder = str(data_folder.resolve())

if not script_path.exists():
    raise FileNotFoundError(f'Nicht gefunden: {script_path}')
if not hparams_root.exists():
    raise FileNotFoundError(f'Nicht gefunden: {hparams_root}')

# Alle Train-Konfigurationen rekursiv einsammeln (inkl. hparams/train.yaml)
hparam_files = sorted({p.resolve() for p in hparams_root.rglob('train*.yaml')})
if selected_files is not None:
    hparam_files = [p for p in hparam_files if p.name in selected_files]

if not hparam_files:
    raise RuntimeError('Keine passenden train*.yaml-Dateien gefunden.')

jobs = [(h, seed) for h in hparam_files for seed in seeds]
if max_runs is not None:
    jobs = jobs[:max_runs]

print(f'Gefundene Train-Dateien: {len(hparam_files)}')
print(f'Geplante Runs (Datei x Seeds): {len(jobs)}')
print('Verwende data_folder:', data_folder)
if override_epochs is not None:
    print('Override number_of_epochs:', override_epochs)

for idx, (hparams_file, seed) in enumerate(jobs, start=1):
    cmd = [
        sys.executable,
        str(script_path),
        str(hparams_file),
        f'data_folder={data_folder}',
        f'seed={seed}',
    ]
    if override_epochs is not None:
        cmd.append(f'number_of_epochs={override_epochs}')

    cmd_str = ' '.join(cmd)
    print(f'[{idx}/{len(jobs)}] {cmd_str}')

    if not dry_run:
        subprocess.run(cmd, check=True, cwd=project_dir)

print('Fertig.')